[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# *Polars* avanzado.

Este *notebook* cubre cuatro temas que se apoyan en los *notebooks* 35 y 36:

1. **Combinación de tablas** (*joins*), incluida la combinación por la fecha más cercana (*as-of join*).
2. **Reestructuración** entre formato largo y ancho con `pivot()` y `unpivot()`.
3. **Modo diferido** (*lazy*): optimización de consultas, motor *streaming* y conjuntos de datos particionados.
4. **SQL** sobre *dataframes* de *Polars*.

Al final se incluye una medición de rendimiento entre *Pandas* y *Polars*.

* La siguiente celda importa las bibliotecas, configura la presentación de flotantes y crea el directorio `data_tmp`.

In [ ]:
import os
import time

import numpy as np
import pandas as pd
import polars as pl

pl.Config.set_fmt_float('full')
dir_tmp = 'data_tmp'
os.makedirs(dir_tmp, exist_ok=True)
print(f"Polars {pl.__version__}")

* La siguiente celda lee los tres archivos de datos de ejemplo (descritos en el *notebook* 34) y calcula el importe neto de cada venta.

In [ ]:
ventas = (
    pl.read_csv(os.path.join('data', 'ventas.csv'), try_parse_dates=True)
    .with_columns(
        importe_neto=(
            pl.col('cantidad') * pl.col('precio_unitario') * (1 - pl.col('descuento').fill_null(0))
        ).round(2)
    )
)
tiendas = pl.read_csv(os.path.join('data', 'tiendas.csv'), try_parse_dates=True)
tipo_cambio = pl.read_csv(os.path.join('data', 'tipo_cambio.csv'), try_parse_dates=True)

print(f"ventas: {ventas.shape} · tiendas: {tiendas.shape} · tipo_cambio: {tipo_cambio.shape}")
tiendas

## Combinación de tablas.

[`df.join()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.join.html) combina dos *dataframes* según los valores de una o más columnas clave.

```
df.join(<otro>, on=<claves>, how=<tipo>, validate=<relación>, coalesce=<booleano>)
```

Donde:

* `<otro>` es el *dataframe* de la derecha.
* `<claves>` son las columnas que deben coincidir. Si se llaman distinto en cada lado, se usan `left_on` y `right_on`.
* `<tipo>` es el tipo de combinación (ver la tabla siguiente). Por omisión es `'inner'`.
* `validate` es opcional y verifica la relación entre las claves: `'1:1'`, `'1:m'`, `'m:1'` o `'m:m'`. Si no se cumple, se desencadena una excepción, lo que evita duplicar renglones por accidente.
* `coalesce=True` combina las columnas clave de ambos lados en una sola. Es útil en `'full'`, donde por omisión se conservan las dos.

| `how=` | Renglones del resultado | Columnas |
| :--- | :--- | :--- |
| `'inner'` | Sólo los que tienen coincidencia en ambos lados. | De ambos. |
| `'left'` | Todos los de la izquierda; nulos donde no hay coincidencia. | De ambos. |
| `'right'` | Todos los de la derecha; nulos donde no hay coincidencia. | De ambos. |
| `'full'` | Todos los de ambos lados; nulos donde no hay coincidencia. | De ambos. |
| `'semi'` | Los de la izquierda que **sí** tienen coincidencia. | Sólo de la izquierda. |
| `'anti'` | Los de la izquierda que **no** tienen coincidencia. | Sólo de la izquierda. |
| `'cross'` | Todas las combinaciones (producto cartesiano). No usa claves. | De ambos. |

Los datos de ejemplo tienen dos inconsistencias intencionales que permiten ver la diferencia entre los tipos: algunas ventas tienen `tienda_id` 99, que no está en el catálogo, y las tiendas 13 y 14 no tienen ventas.

[Guía de *joins*](https://docs.pola.rs/user-guide/transformations/joins/)

**Ejemplos:**

* La siguiente celda combina las ventas con el catálogo de tiendas con `'inner'` y `'left'`. El *inner join* pierde las ventas de la tienda 99; el *left join* las conserva con nulos en las columnas del catálogo.

In [ ]:
tiendas_reducido = tiendas.select('tienda_id', 'nombre', 'region')

inner = ventas.join(tiendas_reducido, on='tienda_id', how='inner')
left = ventas.join(tiendas_reducido, on='tienda_id', how='left', validate='m:1')

print(f"Ventas: {ventas.height:,} · inner: {inner.height:,} · left: {left.height:,}")
left.filter(pl.col('region').is_null()).select('venta_id', 'tienda_id', 'nombre', 'region').head(3)

* La siguiente celda usa `'anti'` en ambos sentidos para encontrar las inconsistencias: ventas cuya tienda no existe en el catálogo y tiendas sin ventas.

In [ ]:
ventas_sin_tienda = ventas.join(tiendas, on='tienda_id', how='anti')
tiendas_sin_ventas = tiendas.join(ventas, on='tienda_id', how='anti')

print(f"Ventas con tienda inexistente: {ventas_sin_tienda.height} (tienda_id: {ventas_sin_tienda['tienda_id'].unique().to_list()})")
tiendas_sin_ventas.select('tienda_id', 'nombre', 'fecha_apertura')

* La siguiente celda usa `'semi'` para obtener las tiendas que vendieron artículos de electrónica el 31 de diciembre de 2025. A diferencia de un *inner join*, cada tienda aparece una sola vez, aunque tenga varias ventas, y sólo con sus propias columnas.

In [ ]:
electronica_31_dic = ventas.filter(
    pl.col('fecha_hora').dt.date() == pl.date(2025, 12, 31),
    pl.col('categoria') == 'Electrónica',
)
print(f"Ventas: {electronica_31_dic.height}")
tiendas.join(electronica_31_dic, on='tienda_id', how='semi').select('tienda_id', 'nombre', 'region')

* La siguiente celda combina el importe total por tienda con el catálogo usando `'full'` y `coalesce=True`: aparecen tanto la tienda 99 (sin datos de catálogo) como las tiendas 13 y 14 (sin importe).

In [ ]:
por_tienda = ventas.group_by('tienda_id').agg(pl.col('importe_neto').sum().round(2).alias('importe'))

por_tienda.join(tiendas_reducido, on='tienda_id', how='full', coalesce=True).sort('tienda_id').tail(4)

* La siguiente celda usa `'cross'` para generar todas las combinaciones de tienda y canal, y les une el importe de cada combinación. Así aparecen también las combinaciones sin ventas (las de las tiendas 13 y 14), que de otro modo faltarían en el resultado.

In [ ]:
canales = ventas.select('canal').unique()

importe_tienda_canal = ventas.group_by('tienda_id', 'canal').agg(
    pl.col('importe_neto').sum().round(2).alias('importe')
)

tiendas_reducido.join(canales, how='cross').join(
    importe_tienda_canal, on=['tienda_id', 'canal'], how='left'
).sort('tienda_id', 'canal').tail(6)

* La siguiente celda intenta combinar las ventas con un catálogo que tiene una tienda repetida, validando que la relación sea `'m:1'`. Se desencadenará una excepción de tipo `ComputeError` porque la clave no es única en el catálogo.

In [ ]:
catalogo_duplicado = pl.concat([tiendas_reducido, tiendas_reducido.head(1)])
ventas.join(catalogo_duplicado, on='tienda_id', how='left', validate='m:1')

### Combinación por la clave más cercana.

[`df.join_asof()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.join_asof.html) combina cada renglón con el renglón de la otra tabla cuya clave es **la más cercana**, en lugar de exigir que sea igual. Es útil con datos de tiempo, por ejemplo para asignar a cada venta el último tipo de cambio publicado.

```
df.join_asof(<otro>, on=<clave>, strategy=<estrategia>, tolerance=<tolerancia>)
```

Donde:

* `<clave>` es una columna ordenable (generalmente una fecha). **Ambos *dataframes* deben estar ordenados por ella.**
* `<estrategia>` es `'backward'` (por omisión: el valor anterior o igual más cercano), `'forward'` (el posterior o igual) o `'nearest'`.
* `<tolerancia>` es opcional y limita la distancia máxima, por ejemplo `'3d'`.

El tipo de cambio sólo se publica en días hábiles. Con un *join* normal, las ventas de fines de semana quedarían sin tipo de cambio; con `join_asof()` y `strategy='backward'` toman el del viernes anterior.

In [ ]:
ventas_usd = (
    ventas
    .with_columns(fecha=pl.col('fecha_hora').dt.date())
    .sort('fecha')
    .join_asof(tipo_cambio.sort('fecha'), on='fecha', strategy='backward')
    .with_columns(importe_usd=(pl.col('importe_neto') / pl.col('usd_mxn')).round(2))
)

# Ventas del sábado 6 y domingo 7 de enero de 2024: usan el tipo de cambio del viernes 5.
ventas_usd.filter(
    pl.col('fecha').is_between(pl.date(2024, 1, 5), pl.date(2024, 1, 7))
).group_by('fecha').agg(pl.col('usd_mxn').first(), pl.len()).sort('fecha').join(
    tipo_cambio, on='fecha', how='left', suffix='_publicado'
)

## Formato largo y formato ancho.

* En **formato largo**, cada renglón es una observación: por ejemplo, una combinación de mes, categoría e importe. Es el formato más cómodo para filtrar, agrupar y graficar.
* En **formato ancho**, los valores de una columna se convierten en columnas: por ejemplo, un renglón por mes y una columna por categoría. Es el formato más cómodo para leer una tabla o compararla visualmente.

| Método | Conversión |
| :--- | :--- |
| [`df.pivot(on=..., index=..., values=..., aggregate_function=...)`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.pivot.html) | De largo a ancho: cada valor distinto de `on` se convierte en una columna; `index` define los renglones; si hay varios valores por celda, se combinan con `aggregate_function` (`'sum'`, `'mean'`, `'first'`, `'len'`, ...). |
| [`df.unpivot(index=..., on=..., variable_name=..., value_name=...)`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.unpivot.html) | De ancho a largo: las columnas de `on` (por omisión, todas las que no están en `index`) se convierten en dos columnas: nombre y valor. |

Las columnas que produce `pivot()` dependen de los datos. Por eso, en el modo diferido (que se estudia a continuación) `pivot()` exige indicar de antemano los valores que serán columnas con el argumento `on_columns`: el *schema* del resultado debe conocerse antes de ejecutar la consulta.

**Ejemplos:**

* La siguiente celda calcula el importe trimestral por categoría en formato largo.

In [ ]:
largo = (
    ventas
    .with_columns(trimestre=pl.col('fecha_hora').dt.year().cast(pl.String) + '-T' + pl.col('fecha_hora').dt.quarter().cast(pl.String))
    .group_by('trimestre', 'categoria')
    .agg(pl.col('importe_neto').sum().round(2).alias('importe'))
    .sort('trimestre', 'categoria')
)
largo.head(6)

* La siguiente celda convierte el resultado a formato ancho: un renglón por trimestre y una columna por categoría.

In [ ]:
ancho = largo.pivot(on='categoria', index='trimestre', values='importe', aggregate_function='sum')
ancho

* La siguiente celda regresa al formato largo con `unpivot()` y verifica que se obtienen los mismos datos.

In [ ]:
regreso = ancho.unpivot(index='trimestre', variable_name='categoria', value_name='importe')
regreso.sort('trimestre', 'categoria').equals(largo)

## Modo diferido (*lazy*).

En el modo inmediato (*eager*), que se ha usado hasta ahora, cada operación se ejecuta en cuanto se escribe. En el **modo diferido**, las operaciones se acumulan en un **plan de consulta** (`pl.LazyFrame`) que sólo se ejecuta al llamar `collect()`. Antes de ejecutarlo, *Polars* revisa el plan completo y lo optimiza.

| Función o método | Descripción |
| :--- | :--- |
| [`df.lazy()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.lazy.html) | Convierte un `DataFrame` en `LazyFrame`. |
| [`pl.scan_csv()`](https://docs.pola.rs/api/python/stable/reference/api/polars.scan_csv.html), [`pl.scan_parquet()`](https://docs.pola.rs/api/python/stable/reference/api/polars.scan_parquet.html), [`pl.scan_ipc()`](https://docs.pola.rs/api/python/stable/reference/api/polars.scan_ipc.html) | Crean un `LazyFrame` a partir de archivos, sin leerlos todavía. |
| [`lf.explain(optimized=<booleano>)`](https://docs.pola.rs/api/python/stable/reference/lazyframe/api/polars.LazyFrame.explain.html) | Muestra el plan de consulta, optimizado (por omisión) o tal como se escribió. |
| [`lf.collect_schema()`](https://docs.pola.rs/api/python/stable/reference/lazyframe/api/polars.LazyFrame.collect_schema.html) | Obtiene el *schema* del resultado sin ejecutar la consulta. |
| [`lf.collect(engine=<motor>)`](https://docs.pola.rs/api/python/stable/reference/lazyframe/api/polars.LazyFrame.collect.html) | Ejecuta el plan y regresa un `DataFrame`. Con `engine='streaming'` procesa los datos por partes. |
| [`lf.sink_parquet()`](https://docs.pola.rs/api/python/stable/reference/lazyframe/api/polars.LazyFrame.sink_parquet.html), `sink_csv()`, `sink_ipc()` | Ejecutan el plan y escriben el resultado directamente a un archivo, sin reunirlo en memoria. |

Un `LazyFrame` admite casi todos los métodos de un `DataFrame` (`select()`, `with_columns()`, `filter()`, `group_by()`, `join()`, `sort()`, ...).

### Optimizaciones.

| Optimización | Efecto |
| :--- | :--- |
| *Predicate pushdown* | Los filtros se aplican lo antes posible; si se lee un archivo, durante la lectura. |
| *Projection pushdown* | Sólo se leen y procesan las columnas que se usan en la consulta. |
| *Slice pushdown* | Si sólo se piden algunos renglones (`head()`), se deja de leer al obtenerlos. |
| Eliminación de subexpresiones comunes | Un cálculo que aparece varias veces se hace una sola vez. |

### ¿Cuándo usar cada modo?

* El modo inmediato es más cómodo para explorar datos paso a paso en un *notebook*.
* El modo diferido conviene cuando se leen archivos grandes, cuando la consulta tiene varios pasos o cuando el resultado no cabe en memoria. Una práctica común es explorar con el modo inmediato y escribir el proceso final con `scan_*()` y `collect()`.

[Guía del modo diferido](https://docs.pola.rs/user-guide/lazy/)

**Ejemplos:**

* La siguiente celda define una consulta diferida sobre el archivo *CSV*: el importe neto en línea por región en 2025. Ninguna operación se ejecuta todavía; el resultado es un `LazyFrame`.

In [ ]:
consulta = (
    pl.scan_csv(os.path.join('data', 'ventas.csv'), try_parse_dates=True)
    .join(pl.scan_csv(os.path.join('data', 'tiendas.csv')), on='tienda_id', how='inner')
    .with_columns(
        importe_neto=pl.col('cantidad') * pl.col('precio_unitario') * (1 - pl.col('descuento').fill_null(0))
    )
    .filter(pl.col('canal') == 'en línea', pl.col('fecha_hora').dt.year() == 2025)
    .group_by('region')
    .agg(pl.col('importe_neto').sum().round(2).alias('importe'))
    .sort('importe', descending=True)
)
type(consulta)

* La siguiente celda obtiene el *schema* del resultado sin ejecutar la consulta.

In [ ]:
consulta.collect_schema()

* La siguiente celda muestra el plan tal como se escribió. Se lee de abajo hacia arriba: primero se leen los archivos completos y el filtro se aplica después del *join*.

In [ ]:
print(consulta.explain(optimized=False))

* La siguiente celda muestra el plan optimizado. Hay que observar dos cambios en la lectura del *CSV* de ventas (`Csv SCAN`): `PROJECT` indica que sólo se leen las columnas necesarias, y `SELECTION` que el filtro se aplica durante la lectura, antes del *join*.

In [ ]:
print(consulta.explain())

* La siguiente celda ejecuta la consulta con `collect()`.

In [ ]:
consulta.collect()

### El motor *streaming*.

Por omisión, `collect()` carga en memoria todos los datos que necesita la consulta. Con `engine='streaming'`, *Polars* procesa los datos en bloques, de modo que puede trabajar con archivos más grandes que la memoria disponible. Las operaciones de `sink_*()` siempre usan este motor y escriben el resultado conforme se calcula.

* La siguiente celda ejecuta la misma consulta con el motor *streaming*. El resultado es el mismo.

In [ ]:
consulta.collect(engine='streaming').equals(consulta.collect())

* La siguiente celda convierte el *CSV* de ventas a *Parquet* con `sink_parquet()`, agregando el importe neto, sin cargar el archivo completo en memoria.

In [ ]:
ruta_parquet = os.path.join(dir_tmp, 'ventas_neto.parquet')

(
    pl.scan_csv(os.path.join('data', 'ventas.csv'), try_parse_dates=True)
    .with_columns(
        importe_neto=(pl.col('cantidad') * pl.col('precio_unitario') * (1 - pl.col('descuento').fill_null(0))).round(2)
    )
    .sink_parquet(ruta_parquet)
)
print(f"{os.path.getsize(ruta_parquet) / 1024:.0f} KB")

### Conjuntos de datos particionados.

Como se vio en el *notebook* 34, un conjunto de datos puede dividirse en carpetas con el esquema de *Hive* (`columna=valor`). `write_parquet(..., partition_by=...)` lo escribe, y `pl.scan_parquet(..., hive_partitioning=True)` lo lee como una sola tabla: un filtro sobre la columna de partición hace que sólo se lean las carpetas necesarias.

* La siguiente celda escribe las ventas particionadas por año y muestra los archivos creados.

In [ ]:
ruta_particiones = os.path.join(dir_tmp, 'ventas_por_anio_polars')

ventas.with_columns(anio=pl.col('fecha_hora').dt.year()).write_parquet(ruta_particiones, partition_by='anio')

for carpeta, _, archivos in sorted(os.walk(ruta_particiones)):
    if archivos:
        print(carpeta, archivos)

* La siguiente celda consulta sólo 2025. En el plan optimizado, el filtro sobre `anio` se aplica a nivel de archivos.

In [ ]:
por_anio = (
    pl.scan_parquet(ruta_particiones, hive_partitioning=True)
    .filter(pl.col('anio') == 2025)
    .select(pl.len().alias('ventas_2025'), pl.col('importe_neto').sum().round(2).alias('importe'))
)
por_anio.collect()

## SQL en *Polars*.

*Polars* puede ejecutar consultas *SQL* sobre *dataframes* y *lazyframes*. Las consultas se traducen al mismo plan que las expresiones, así que se optimizan igual.

| Forma | Descripción |
| :--- | :--- |
| [`df.sql(<consulta>)`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.sql.html) | Consulta sobre un solo *dataframe*, al que se hace referencia como `self`. |
| [`pl.SQLContext(<nombre>=<frame>, ...)`](https://docs.pola.rs/api/python/stable/reference/sql/python_api.html) | Registra varios *dataframes* o *lazyframes* con un nombre de tabla; `execute(<consulta>)` regresa un `LazyFrame` (o un `DataFrame` con `eager=True`). |

Es útil para equipos que ya conocen *SQL* o para migrar consultas existentes. El dialecto es similar al de *PostgreSQL*.

[Guía de *SQL*](https://docs.pola.rs/user-guide/sql/intro/)

**Ejemplos:**

* La siguiente celda calcula con `df.sql()` el ticket promedio por canal.

In [ ]:
ventas.sql('''
    SELECT canal, COUNT(*) AS ventas, ROUND(AVG(importe_neto), 2) AS ticket_promedio
    FROM self
    GROUP BY canal
    ORDER BY ventas DESC
''')

* La siguiente celda registra las ventas y el catálogo en un `SQLContext` y obtiene las tres tiendas con mayor importe en 2025, combinando ambas tablas.

In [ ]:
contexto = pl.SQLContext(ventas=ventas.lazy(), tiendas=tiendas.lazy())

contexto.execute('''
    SELECT t.nombre, t.region, ROUND(SUM(v.importe_neto), 2) AS importe
    FROM ventas AS v
    INNER JOIN tiendas AS t ON v.tienda_id = t.tienda_id
    WHERE EXTRACT(YEAR FROM v.fecha_hora) = 2025
    GROUP BY t.nombre, t.region
    ORDER BY importe DESC
    LIMIT 3
''').collect()

## Medición de rendimiento: *Pandas* y *Polars*.

Comparar bibliotecas requiere cuidado para que la medición sea justa:

* **Mismos datos y misma operación.** Se generan una sola vez y ambos *dataframes* se construyen **antes** de medir, porque lo que interesa es el tiempo de la operación, no el de la carga.
* **Varias repeticiones.** La primera ejecución puede ser más lenta (por ejemplo, por la memoria caché), así que se toma el menor de varios tiempos.
* **Resultados que dependen del equipo.** *Polars* usa todos los núcleos del procesador y *Pandas* casi siempre uno solo, así que la diferencia cambia según la máquina. En *GitHub Codespaces*, la máquina más pequeña tiene 2 núcleos.

La operación es típica de un análisis: filtrar, agrupar por una columna de texto y calcular dos estadísticas.

* La siguiente celda genera 5 millones de renglones con *NumPy* y construye ambos *dataframes*. Esta parte no se mide.

In [ ]:
n = 5_000_000
rng = np.random.default_rng(37)
datos = {
    'tienda': rng.choice([f'T{i:02d}' for i in range(1, 51)], n),
    'cantidad': rng.integers(1, 6, n),
    'precio': rng.uniform(50, 2000, n).round(2),
}

df_pandas = pd.DataFrame(datos)
df_polars = pl.DataFrame(datos)
print(f"{n:,} renglones · {os.cpu_count()} núcleos disponibles")

* La siguiente celda define una función que ejecuta una operación varias veces y regresa el menor tiempo, y las tres versiones de la operación: *Pandas*, *Polars* en modo inmediato y *Polars* en modo diferido.

In [ ]:
def mejor_tiempo(funcion, repeticiones=3):
    tiempos = []
    for _ in range(repeticiones):
        inicio = time.perf_counter()
        funcion()
        tiempos.append(time.perf_counter() - inicio)
    return min(tiempos)


def con_pandas():
    filtrado = df_pandas[df_pandas['cantidad'] >= 2]
    return filtrado.assign(importe=filtrado['cantidad'] * filtrado['precio']).groupby('tienda').agg(
        importe_total=('importe', 'sum'), precio_medio=('precio', 'mean')
    )


def con_polars():
    return (
        df_polars
        .filter(pl.col('cantidad') >= 2)
        .group_by('tienda')
        .agg(
            (pl.col('cantidad') * pl.col('precio')).sum().alias('importe_total'),
            pl.col('precio').mean().alias('precio_medio'),
        )
    )


def con_polars_lazy():
    return (
        df_polars.lazy()
        .filter(pl.col('cantidad') >= 2)
        .group_by('tienda')
        .agg(
            (pl.col('cantidad') * pl.col('precio')).sum().alias('importe_total'),
            pl.col('precio').mean().alias('precio_medio'),
        )
        .collect()
    )

* La siguiente celda verifica que las tres versiones dan el mismo resultado antes de medir. Se compara con `np.allclose()` porque las sumas de punto flotante pueden diferir en los últimos decimales según el orden en que se suman.

In [ ]:
referencia = con_polars().sort('tienda')
resultados = {
    'pandas': pl.from_pandas(con_pandas().reset_index()).sort('tienda'),
    'polars diferido': con_polars_lazy().sort('tienda'),
}
for nombre, resultado in resultados.items():
    iguales = all(
        np.allclose(referencia[columna], resultado[columna]) for columna in ['importe_total', 'precio_medio']
    )
    print(f"{nombre}: {iguales}")

* La siguiente celda mide las tres versiones. Los tiempos y la proporción varían de un equipo a otro.

In [ ]:
t_pandas = mejor_tiempo(con_pandas)
t_polars = mejor_tiempo(con_polars)
t_lazy = mejor_tiempo(con_polars_lazy)

print(f"Pandas:                  {t_pandas:.3f} s")
print(f"Polars (inmediato):      {t_polars:.3f} s  ({t_pandas / t_polars:.1f} veces más rápido)")
print(f"Polars (diferido):       {t_lazy:.3f} s  ({t_pandas / t_lazy:.1f} veces más rápido)")

La diferencia proviene de varias fuentes: *Polars* reparte el filtro y la agrupación entre los núcleos disponibles, evita crear columnas intermedias (el importe se calcula dentro de la agregación) y trabaja directamente sobre el formato de *Arrow*. Con pocos datos o con un solo núcleo, la diferencia se reduce.

## Resumen.

| Concepto | Descripción |
| :--- | :--- |
| ***Joins*** | `inner`, `left`, `right`, `full`, `semi`, `anti` y `cross`; `validate` detecta claves duplicadas. |
| **`join_asof()`** | Combina con la clave más cercana; ambos lados deben estar ordenados. |
| **`pivot()` y `unpivot()`** | Conversión entre formato largo y ancho. |
| **Modo diferido** | `LazyFrame`, `scan_*()`, `explain()` y `collect()`; optimiza filtros, columnas y renglones antes de ejecutar. |
| **Motor *streaming*** | `collect(engine='streaming')` y `sink_*()` procesan los datos por partes. |
| **Particiones** | `write_parquet(partition_by=...)` y `scan_parquet(hive_partitioning=True)` leen sólo las carpetas necesarias. |
| ***SQL*** | `df.sql()` y `pl.SQLContext` traducen consultas *SQL* al mismo plan optimizado. |
| **Medición de rendimiento** | Mismos datos ya cargados, varias repeticiones y conciencia de que el resultado depende del equipo. |

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>